In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
data_path = Path("../data/processed/clean_womens_shoes.parquet")

shoes = pd.read_parquet(data_path)

print("Dataset shape:", shoes.shape)
shoes.head()

Dataset shape: (53176, 10)


,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description,shoe_type,condition_group
0,14,HOLD for Dogs2016 Minnetonka boots,3,Women/Shoes/Boots,UGG Australia,43.0,0,Authentic. Suede fringe boots. Great condition...,Boots,Condition 3
1,70,Adidas Ultraboost Shoes,3,Women/Shoes/Athletic,Adidas,61.0,0,Overall good condition. A few signs of wear,Athletic,Condition 3
2,107,Boots NWT 6.5,1,Women/Shoes/Boots,Merona,13.0,1,Merona short boot new with tag size 6.5 come j...,Boots,Condition 1
3,108,New Duck Boots sz.7.5,1,Women/Shoes/Boots,Boulevard Boutique,38.0,0,New Duck Boots Sz.7.5 Stock up on These Trendy...,Boots,Condition 1
4,115,Steve Madden wedges,2,Women/Shoes/Pumps,Steve Madden,25.0,1,Never worn!!! Brown leather strap wedges!,Pumps,Condition 2


In [3]:
shoes["peer_median_price"] = (
    shoes
    .groupby(["shoe_type", "condition_group"])["price"]
    .transform("median")
)

shoes[
    [
        "brand_name",
        "shoe_type",
        "condition_group",
        "price",
        "peer_median_price"
    ]
].head(10)

,brand_name,shoe_type,condition_group,price,peer_median_price
0,UGG Australia,Boots,Condition 3,43.0,36.0
1,Adidas,Athletic,Condition 3,61.0,28.0
2,Merona,Boots,Condition 1,13.0,61.0
3,Boulevard Boutique,Boots,Condition 1,38.0,61.0
4,Steve Madden,Pumps,Condition 2,25.0,21.0
5,Manolo Blahnik,Pumps,Condition 4–5,9.0,15.0
6,GUESS,Sandals,Condition 1,31.0,29.0
7,Banana Republic,Boots,Condition 2,21.0,40.0
8,JustFab,Boots,Condition 1,31.0,61.0
9,Crocs,Loafers & Slip-Ons,Condition 3,16.0,20.0


In [4]:
shoes["adjusted_price_ratio"] = (
    shoes["price"] / shoes["peer_median_price"]
)

shoes[
    [
        "brand_name",
        "shoe_type",
        "condition_group",
        "price",
        "peer_median_price",
        "adjusted_price_ratio"
    ]
].head(10)

,brand_name,shoe_type,condition_group,price,peer_median_price,adjusted_price_ratio
0,UGG Australia,Boots,Condition 3,43.0,36.0,1.194444
1,Adidas,Athletic,Condition 3,61.0,28.0,2.178571
2,Merona,Boots,Condition 1,13.0,61.0,0.213115
3,Boulevard Boutique,Boots,Condition 1,38.0,61.0,0.622951
4,Steve Madden,Pumps,Condition 2,25.0,21.0,1.190476
5,Manolo Blahnik,Pumps,Condition 4–5,9.0,15.0,0.600000
6,GUESS,Sandals,Condition 1,31.0,29.0,1.068966
7,Banana Republic,Boots,Condition 2,21.0,40.0,0.525000
8,JustFab,Boots,Condition 1,31.0,61.0,0.508197
9,Crocs,Loafers & Slip-Ons,Condition 3,16.0,20.0,0.800000


In [5]:
brand_summary = (
    shoes
    .groupby("brand_name")
    .agg(
        listing_count=("price", "size"),
        median_raw_price=("price", "median"),
        median_adjusted_ratio=("adjusted_price_ratio", "median")
    )
    .reset_index()
)

brand_summary.sort_values(
    "listing_count",
    ascending=False
).head(20)

,brand_name,listing_count,median_raw_price,median_adjusted_ratio
507,Nike,8969,36.0,1.047619
673,UGG Australia,3304,55.0,1.475000
183,Converse,3204,26.0,0.935484
682,VANS,2445,26.0,0.920000
15,Adidas,1965,51.0,1.464286
662,Tory Burch,1721,61.0,2.842105
628,Steve Madden,1680,24.0,0.918033
468,Michael Kors,1416,31.0,1.285714
176,Coach,1391,26.0,1.083333
527,PINK,1160,29.0,1.043810


In [6]:
eligible_brands = brand_summary[
    brand_summary["listing_count"] >= 50
].copy()

print("Eligible brands:", len(eligible_brands))

eligible_brands[
    [
        "brand_name",
        "listing_count",
        "median_raw_price",
        "median_adjusted_ratio"
    ]
].sort_values(
    "median_adjusted_ratio",
    ascending=False
).head(20)

Eligible brands: 116


,brand_name,listing_count,median_raw_price,median_adjusted_ratio
651,Tieks,59,156.0,7.947368
169,Christian Louboutin,301,156.0,7.733333
683,Valentino,55,156.0,7.052632
156,Chanel,102,106.0,4.292857
630,Stuart Weitzman,56,150.0,4.263333
303,Gucci,172,87.0,3.773183
129,Burberry,87,76.0,3.027778
662,Tory Burch,1721,61.0,2.842105
154,Chaco,281,56.0,2.666667
544,Prada,55,56.0,2.650000


In [7]:
q25 = eligible_brands["median_adjusted_ratio"].quantile(0.25)
q75 = eligible_brands["median_adjusted_ratio"].quantile(0.75)

print("25th percentile cutoff:", q25)
print("75th percentile cutoff:", q75)

25th percentile cutoff: 0.6666666666666666
75th percentile cutoff: 1.2783333333333333


In [8]:
def assign_tier(ratio):
    if ratio <= q25:
        return "Value"
    elif ratio >= q75:
        return "Premium"
    else:
        return "Mid-Market"

eligible_brands["price_tier"] = (
    eligible_brands["median_adjusted_ratio"]
    .apply(assign_tier)
)

eligible_brands["price_tier"].value_counts()

price_tier
Mid-Market    57
Value         30
Premium       29
Name: count, dtype: int64

In [9]:
from pathlib import Path

brand_tier_path = Path("../data/processed/brand_tiers.parquet")
listing_path = Path("../data/processed/shoes_with_adjusted_prices.parquet")

eligible_brands.to_parquet(
    brand_tier_path,
    index=False
)

shoes.to_parquet(
    listing_path,
    index=False
)

print("Saved brand tiers to:", brand_tier_path)
print("Saved adjusted listings to:", listing_path)
print("Number of eligible brands:", len(eligible_brands))
print("Number of listings:", len(shoes))

Saved brand tiers to: ../data/processed/brand_tiers.parquet
Saved adjusted listings to: ../data/processed/shoes_with_adjusted_prices.parquet
Number of eligible brands: 116
Number of listings: 53176
